In [57]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv

from ultralytics.utils.checks import check_imshow
from ultralytics.utils.plotting import Annotator, colors

from collections import defaultdict

In [ ]:
# COLORS = sv.ColorPalette.from_hex(["#E6194B", "#3CB44B", "#FFE119", "#3C76D1"])
COLORS = sv.ColorPalette.from_hex(["#00FF00", "#FF0000"])

ZONE_IN_POLYGONS = [
    np.array([[756, 1042], [865, 1048], [1017, 914], [913, 864]]),
    np.array([[1719, 676], [1728, 610], [1526, 520], [1485, 600]]),
    np.array([[1046, 301], [992, 267], [1091, 156], [1158, 170]]),
    np.array([[390, 421], [535, 468], [467, 571], [340, 506]]),
]

ZONE_OUT_POLYGONS = [
    np.array([[597, 1028], [706, 1044], [724, 812], [623, 820]]),
    np.array([[1464, 765], [1709, 781], [1720, 704], [1483, 665]]),
    np.array([[1160, 315], [1250, 321], [1244, 167], [1171, 166]]),
    np.array([[594, 363], [591, 412], [398, 403], [404, 346]]),
]

model = YOLO("y_v11_100e_b07.pt")
class_names = model.model.names

names = list(class_names.values())

# inicializo en cero los dos arrays (de entrada y salida) que tendrán la cantidad de objetos finales por zona y por clase
"""
Example:
    {
        0: {
            "bicycle": 0,
            "bus": 0,
            "car": 0,
            "motorbike": 0,
            "truck": 0,
            "van": 0
        },
        .
        .
        .
        3: {
            "bicycle": 0,
            "bus": 0,
            "car": 0,
            "motorbike": 0,
            "truck": 0,
            "van": 0
        }
    }
"""
total_obj_zone_in = { i: {key: 0 for key in names} for i in range(len(ZONE_IN_POLYGONS)) }
total_obj_zone_out = { i: {key: 0 for key in names} for i in range(len(ZONE_OUT_POLYGONS)) }

# classes = {
#     "car": "Auto",
#     "bus": "Colectivo",
#     "truck": "Camión",
#     "van": "Camioneta",
#     "motorbike": "Moto",
#     "bicycle": "Bicicleta",
# }

In [59]:
obj_zones_out = []
obj_in_out_zones = defaultdict(lambda: [])

track_history = defaultdict(lambda: [])
data_obj_history = defaultdict(lambda: [])

def get_center_bb(box):
    x_center = int((box[0] + box[2]) / 2)
    y_center = int((box[1] + box[3]) / 2)
    return (x_center, y_center)

def draw_polygons(annotated_frame, polygon, number_polygon, zone_type, thickness):
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1
    cv2.polylines(
        annotated_frame, [polygon], isClosed=True, color=COLORS.colors[zone_type].as_bgr(), thickness=thickness
    )
    zone_center = sv.get_polygon_center(polygon=polygon)
    cv2.putText(annotated_frame, str(number_polygon), (int(zone_center.x), int(zone_center.y)), font, font_scale, COLORS.colors[zone_type].as_bgr(), thickness=thickness)
    
    return annotated_frame

def draw_zones_in_out(annotated_frame, thickness):
    for i, (zone_in, zone_out) in enumerate(zip(ZONE_IN_POLYGONS, ZONE_OUT_POLYGONS)):
        draw_polygons(annotated_frame, zone_in, i, 0, thickness)
        draw_polygons(annotated_frame, zone_out, i, 1, thickness)
    return annotated_frame

def update_results_zones(box, track_id):
    if (track_id in obj_in_out_zones and track_id not in obj_zones_out):
        for i, (polygon) in enumerate(ZONE_OUT_POLYGONS):
            if (cv2.pointPolygonTest(polygon, get_center_bb(box), False) > 0):  # > 0 dentro del polígono
                obj_zones_out.append(track_id)
                # del calculo de la entropia tengo que obtener la probabilidad para asignarle una clase al objeto
                # entropy = calculate_entropy(track_id)

                ult_history_element = data_obj_history[track_id][len(data_obj_history[track_id]) - 1]
                class_name = ult_history_element["class"]["name"]
                track_zone_in = obj_in_out_zones[track_id]["in"]
                total_obj_zone_in[track_zone_in][class_name] += 1
                total_obj_zone_out[i][class_name] += 1
                
                break
    elif (track_id not in obj_in_out_zones):
        for i, (polygon) in enumerate(ZONE_IN_POLYGONS):
            if (cv2.pointPolygonTest(polygon, get_center_bb(box), False) > 0):  # > 0 dentro del polígono
                obj_in_out_zones.setdefault(track_id, {})
                obj_in_out_zones[track_id].setdefault("in", i)
                break

                
def draw_bb_and_save_track(frame, annotator, box, cls, track_id, act_frame, confidence):
    annotator.box_label(box, color=colors(int(cls), True), label=f"{track_id} - {class_names[int(cls)]}")

    # Store tracking and data object history
    if (track_id not in obj_zones_out):
        data_obj_history[track_id].append(
            {
                "act_frame": act_frame,
                "class": {
                    "id": cls,
                    "name": class_names[int(cls)]
                },
                "confidence": confidence
            })
    else:
        entropy = calculate_entropy(track_id)
    track = track_history[track_id]
    track.append((int((box[0] + box[2]) / 2), int((box[1] + box[3]) / 2)))
    if len(track) > 30:
        track.pop(0)

    # Plot tracks
    points = np.array(track, dtype=np.int32).reshape((-1, 1, 2))
    cv2.polylines(frame, [points], isClosed=False, color=colors(int(cls), True), thickness=2)

def calculate_entropy(track_id):
    return track_id

In [62]:
video_path = "otro_minuto.mp4"
cap = cv2.VideoCapture(video_path)

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

# result = cv2.VideoWriter("object_tracking.avi",
#                        cv2.VideoWriter_fourcc(*'mp4v'),
#                        fps,
#                        (w, h))

# inicializo en cero los arays que tendrán la cantidad de objetos finales por zona

act_frame = 0
while cap.isOpened():
    success, frame = cap.read()
    act_frame += 1

    if success:
        results = model.track(frame, persist=True, verbose=False)
        boxes = results[0].boxes.xyxy.cpu()

        draw_zones_in_out(frame, 2)
        
        if results[0].boxes.id is not None:
            clss = results[0].boxes.cls.cpu().tolist()
            track_ids = results[0].boxes.id.int().cpu().tolist()
            confs = results[0].boxes.conf.float().cpu().tolist()
            # Annotator Init
            annotator = Annotator(frame, line_width=1)
            for box, cls, track_id, confidence in zip(boxes, clss, track_ids, confs):
                update_results_zones(box, track_id)
                if (track_id in obj_in_out_zones):
                    draw_bb_and_save_track(frame, annotator, box, cls, track_id, act_frame, confidence)

        cv2.imshow("Video", frame)
        # result.write(frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        break

print(total_obj_zone_in)
print(total_obj_zone_out)
# result.release()
cap.release()
cv2.destroyAllWindows()

{0: {'bicycle': 0, 'bus': 0, 'car': 9, 'motorbike': 0, 'truck': 1, 'van': 0}, 1: {'bicycle': 0, 'bus': 0, 'car': 9, 'motorbike': 0, 'truck': 2, 'van': 2}, 2: {'bicycle': 0, 'bus': 0, 'car': 10, 'motorbike': 0, 'truck': 0, 'van': 0}, 3: {'bicycle': 0, 'bus': 0, 'car': 4, 'motorbike': 0, 'truck': 0, 'van': 0}}
{0: {'bicycle': 0, 'bus': 0, 'car': 5, 'motorbike': 0, 'truck': 1, 'van': 0}, 1: {'bicycle': 0, 'bus': 0, 'car': 10, 'motorbike': 0, 'truck': 0, 'van': 0}, 2: {'bicycle': 0, 'bus': 0, 'car': 10, 'motorbike': 0, 'truck': 0, 'van': 1}, 3: {'bicycle': 0, 'bus': 0, 'car': 7, 'motorbike': 0, 'truck': 2, 'van': 1}}


In [61]:
for i, (zone) in enumerate(total_obj_zone_in):
    print(f"Zona de entrada {i}")
    for key, value in total_obj_zone_in[i].items():
        print(f"    {key}: {value}")

for i, (zone) in enumerate(total_obj_zone_out):
    print(f"Zona de salida {i}")
    for key, value in total_obj_zone_out[i].items():
        print(f"    {key}: {value}")

Zona de entrada 0
    bicycle: 0
    bus: 0
    car: 9
    motorbike: 0
    truck: 1
    van: 0
Zona de entrada 1
    bicycle: 0
    bus: 0
    car: 9
    motorbike: 0
    truck: 2
    van: 2
Zona de entrada 2
    bicycle: 0
    bus: 0
    car: 10
    motorbike: 0
    truck: 0
    van: 0
Zona de entrada 3
    bicycle: 0
    bus: 0
    car: 4
    motorbike: 0
    truck: 0
    van: 0
Zona de salida 0
    bicycle: 0
    bus: 0
    car: 5
    motorbike: 0
    truck: 1
    van: 0
Zona de salida 1
    bicycle: 0
    bus: 0
    car: 10
    motorbike: 0
    truck: 0
    van: 0
Zona de salida 2
    bicycle: 0
    bus: 0
    car: 10
    motorbike: 0
    truck: 0
    van: 1
Zona de salida 3
    bicycle: 0
    bus: 0
    car: 7
    motorbike: 0
    truck: 2
    van: 1
